## 05.Topic Modeling with TF-IDF & K-Means

### 학습 목표
1. **지도학습(분류)** 과 **비지도학습(군집·토픽)** 의 차이를 이해한다.
2. Amazon Comprehend의 **관리형 토픽 모델링** API 흐름을 개요로 파악한다.
3. **KMeans 군집화**로 라벨 없이 토픽이 자동으로 묶이는 과정을 직접 체험한다.

### 지도학습 vs 비지도학습
| 구분 | 지도학습 (앞의 실습들) | 비지도학습 (이번 실습) |
|---|---|---|
| 정답 라벨 | 있음 (긍정/부정 등) | **없음** |
| 하는 일 | 분류·예측 | **비슷한 것끼리 묶기(군집)** |
| 예시 | 감성 분석, 개체 인식 | **토픽 모델링, 클러스터링** |

> 토픽 모델링 = "이 많은 문서에 *어떤 주제들이* 있는지" 라벨 없이 자동으로 발견하는 것.

In [ ]:
#환경 초기화
import boto3, sys, subprocess
import pandas as pd
import matplotlib.pyplot as plt

# 한글 폰트 + scikit-learn
try:
    import koreanize_matplotlib
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'koreanize-matplotlib'])
    import koreanize_matplotlib
try:
    import sklearn
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'])

comprehend = boto3.client('comprehend', region_name='ap-northeast-2')
print('환경 초기화 완료')

In [ ]:
# Amazon Comprehend 관리형 토픽 모델링
# [참고 코드] Comprehend 토픽 모델링은 이렇게 '비동기 잡'으로 실행됨 (실행 X)

def run_comprehend_topic_modeling(input_s3, output_s3, role_arn, num_topics=10):
    response = comprehend.start_topics_detection_job(
        InputDataConfig={'S3Uri': input_s3, 'InputFormat': 'ONE_DOC_PER_LINE'},
        OutputDataConfig={'S3Uri': output_s3},
        DataAccessRoleArn=role_arn,
        NumberOfTopics=num_topics,
    )
    return response['JobId']

# 결과물(완료 후 output S3):
#   - topic-terms.csv : 토픽별 핵심 단어
#   - doc-topics.csv  : 문서별로 배정된 토픽
print('참고: Comprehend 토픽 모델링은 S3 + 비동기 잡으로 동작합니다 (수십 분 소요).')
print('아래에서는 같은 개념(비지도 군집)을 즉시 체험합니다.')

In [ ]:
# 문서 벡터화 (TF-IDF)

from sklearn.feature_extraction.text import TfidfVectorizer

# 토픽이 또렷한 샘플 리뷰 (배송 / 품질 / 가격·환불 / 상담)
sample_docs = [
    '배송 속도가 느리고 배송이 일주일 걸렸어요',
    '배송 박스가 찌그러져서 왔어요',
    '배송 빨라서 좋고 당일 배송 편리해요',
    '배송 기사님 친절하고 배송 상태 좋아요',
    '제품 품질이 사진과 달라 품질 실망이에요',
    '원단 품질 훌륭하고 품질 튼튼해요',
    '품질 금방 망가지고 품질 별로예요',
    '마감 품질 깔끔하고 품질 만족해요',
    '가격 비싸고 환불 원하는데 환불 안돼요',
    '가격 합리적이고 가격 가성비 좋아요',
    '환불 절차 복잡하고 환불 느려요',
    '할인 가격 저렴해서 가격 만족해요',
    '상담 직원이 불친절하고 상담 별로예요',
    '상담 답변 빠르고 상담 친절해요',
    '상담 연결 오래 걸리고 상담 불편해요',
    '고객센터 상담 만족하고 상담 좋아요',
]

# 일반적인 단어(불용어)는 토픽 구분에 도움이 안 되므로 제거
STOPWORDS = ['너무','정말','아주','매우','그냥','좋아요','좋고','걸렸어요','걸리고',
             '왔어요','만족해요','편리해요','친절해요','달라','안돼요','원하는데',
             '저렴해서','빠르고','느리고','느려요','복잡하고','깔끔하고','훌륭하고',
             '튼튼해요','불친절하고','별로예요','실망이에요','망가지고','찌그러져서']

vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", stop_words=STOPWORDS)
X = vectorizer.fit_transform(sample_docs) 
print('문서 수:', X.shape[0], '| 단어(특성) 수:', X.shape[1])

In [ ]:
# KMeans 군집화

from sklearn.cluster import KMeans

NUM_TOPICS = 4
km = KMeans(n_clusters=NUM_TOPICS, random_state=42, n_init=10) 
labels = km.fit_predict(X) 

print('문서별 자동 배정 토픽 (라벨 없이 묶인 결과):')
print('-' * 50)
for i, doc in enumerate(sample_docs):
    print(f'  [토픽 {labels[i]}]  {doc}')

In [ ]:
# 토픽별 대표 단어 + 군집 크기 시각화

import numpy as np
terms = vectorizer.get_feature_names_out()

print('🔑 토픽별 대표 단어 (군집 중심에서 가중치 높은 단어):')
for c in range(NUM_TOPICS):
    top_idx = km.cluster_centers_[c].argsort()[::-1][:5]
    print(f'  토픽 {c}: {[terms[i] for i in top_idx]}')

counts = pd.Series(labels).value_counts().sort_index()
plt.figure(figsize=(7, 4))
plt.bar([f'토픽 {i}' for i in counts.index], counts.values, color='#3498db')
plt.title('토픽(군집)별 리뷰 수')
plt.ylabel('문서 수')
plt.tight_layout()
plt.show()

In [ ]:
# vectorizer로 변환 후 토픽 예측

new_reviews = [
    '배송 상자가 또 찌그러져서 왔어요',
    '환불 절차가 복잡하고 가격도 비싸요',
    '상담 직원이 빠르게 답변했어요',
]

new_X = vectorizer.transform(new_reviews) 
pred  = km.predict(new_X)

for review, topic in zip(new_reviews, pred):
    print(f'  [토픽 {topic}]  {review}')